# 📱 Smartphone, Sono e Estresse
## Como o uso de tecnologia afeta nosso bem-estar?

> **Objetivo:** Explorar padrões de uso de smartphones e seu impacto na saúde e produtividade de 50.000 usuários.

---
**Dataset:** Smartphone Usage & Productivity Dataset (50k registros)  
**Ferramentas:** Python · Pandas · Matplotlib · Seaborn  
**Autor:** Jack Baptista

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'

df = pd.read_csv('Smartphone_Usage_Productivity_Dataset_50000.csv')
print(f'✅ Dataset carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
df.head()

---
## 🔍 Ato 1 — Quem são esses usuários?
*Entendendo o perfil demográfico antes de qualquer análise.*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Perfil Demográfico dos Usuários', fontsize=16, fontweight='bold', y=1.02)

# Distribuição de idade
axes[0].hist(df['Age'], bins=20, color=PALETTE[0], edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribuição de Idade')
axes[0].set_xlabel('Idade')
axes[0].set_ylabel('Frequência')
axes[0].axvline(df['Age'].mean(), color='red', linestyle='--', label=f'Média: {df["Age"].mean():.1f}')
axes[0].legend()

# Gênero
gender_counts = df['Gender'].value_counts()
axes[1].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
            colors=PALETTE[:2], startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Distribuição por Gênero')

# Ocupação
occ_counts = df['Occupation'].value_counts()
axes[2].barh(occ_counts.index, occ_counts.values, color=PALETTE)
axes[2].set_title('Distribuição por Ocupação')
axes[2].set_xlabel('Quantidade')
for i, v in enumerate(occ_counts.values):
    axes[2].text(v + 100, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('01_perfil_demografico.png', bbox_inches='tight')
plt.show()
print('\n💡 Insight: A base é distribuída entre adultos de 18 a 60 anos, com representação equilibrada entre gêneros e múltiplas ocupações.')

---
## 📱 Ato 2 — Como eles usam o celular?
*Padrões de comportamento digital.*

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Padrões de Uso de Smartphone', fontsize=16, fontweight='bold')

# Uso diário — histograma
axes[0,0].hist(df['Daily_Phone_Hours'], bins=25, color=PALETTE[0], edgecolor='white')
axes[0,0].axvline(df['Daily_Phone_Hours'].mean(), color='red', linestyle='--',
                  label=f'Média: {df["Daily_Phone_Hours"].mean():.1f}h')
axes[0,0].set_title('Horas de Uso Diário')
axes[0,0].set_xlabel('Horas/dia')
axes[0,0].legend()

# Redes sociais por ocupação
sm_occ = df.groupby('Occupation')['Social_Media_Hours'].mean().sort_values(ascending=True)
axes[0,1].barh(sm_occ.index, sm_occ.values, color=PALETTE[1])
axes[0,1].set_title('Média de Redes Sociais por Ocupação')
axes[0,1].set_xlabel('Horas/dia')
for i, v in enumerate(sm_occ.values):
    axes[0,1].text(v + 0.03, i, f'{v:.2f}h', va='center', fontsize=9)

# Fim de semana vs dia útil
axes[1,0].scatter(df['Daily_Phone_Hours'], df['Weekend_Screen_Time_Hours'],
                  alpha=0.05, color=PALETTE[2], s=5)
axes[1,0].set_title('Uso Diário vs Fim de Semana')
axes[1,0].set_xlabel('Horas/dia (semana)')
axes[1,0].set_ylabel('Horas/dia (fim de semana)')
corr = df['Daily_Phone_Hours'].corr(df['Weekend_Screen_Time_Hours'])
axes[1,0].text(0.05, 0.92, f'r = {corr:.2f}', transform=axes[1,0].transAxes,
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Device type
device_counts = df['Device_Type'].value_counts()
axes[1,1].pie(device_counts, labels=device_counts.index, autopct='%1.1f%%',
              colors=PALETTE[:2], wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1,1].set_title('Distribuição por Tipo de Dispositivo')

plt.tight_layout()
plt.savefig('02_uso_smartphone.png', bbox_inches='tight')
plt.show()
print(f'\n💡 Insight: A média de {df["Daily_Phone_Hours"].mean():.1f}h/dia é expressiva. '
      f'A correlação de {corr:.2f} entre semana e fim de semana sugere que o hábito é consistente.')

---
## 😴 Ato 3 — Qual o impacto na saúde?
*A relação entre tela, sono e estresse — o coração da análise.*

In [ ]:
# Mapa de calor de correlações
numeric_cols = ['Daily_Phone_Hours', 'Social_Media_Hours', 'Sleep_Hours',
                'Stress_Level', 'Work_Productivity_Score', 'App_Usage_Count',
                'Caffeine_Intake_Cups', 'Weekend_Screen_Time_Hours', 'Age']

fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Mapa de Correlação entre Variáveis', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('03_correlacao.png', bbox_inches='tight')
plt.show()
print('\n💡 Leia o mapa: células verdes = correlação positiva, vermelhas = negativa. Identifique quais pares têm relações mais fortes.')

In [ ]:
# Sono e Estresse por faixa de uso de tela
df['Screen_Category'] = pd.cut(df['Daily_Phone_Hours'],
                                bins=[0, 3, 6, 9, 12],
                                labels=['Baixo\n(0-3h)', 'Moderado\n(3-6h)',
                                        'Alto\n(6-9h)', 'Muito Alto\n(9-12h)'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Impacto do Uso de Tela na Saúde', fontsize=15, fontweight='bold')

# Sono por categoria de uso
sns.boxplot(data=df, x='Screen_Category', y='Sleep_Hours',
            palette=PALETTE, ax=axes[0])
axes[0].set_title('Sono por Categoria de Uso de Tela')
axes[0].set_xlabel('Uso Diário de Tela')
axes[0].set_ylabel('Horas de Sono')
axes[0].axhline(7, color='green', linestyle='--', alpha=0.7, label='Mínimo recomendado (7h)')
axes[0].legend()

# Estresse por categoria de uso
sns.boxplot(data=df, x='Screen_Category', y='Stress_Level',
            palette=PALETTE, ax=axes[1])
axes[1].set_title('Estresse por Categoria de Uso de Tela')
axes[1].set_xlabel('Uso Diário de Tela')
axes[1].set_ylabel('Nível de Estresse (1-10)')

plt.tight_layout()
plt.savefig('04_saude_vs_tela.png', bbox_inches='tight')
plt.show()

# Estatísticas por categoria
summary = df.groupby('Screen_Category', observed=True)[['Sleep_Hours', 'Stress_Level']].mean().round(2)
print('\n📊 Médias por categoria de uso:')
print(summary)

In [ ]:
# Estresse médio por ocupação — ranking
stress_occ = df.groupby('Occupation')['Stress_Level'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(stress_occ.index, stress_occ.values,
              color=[PALETTE[0] if v == stress_occ.max() else PALETTE[2] for v in stress_occ.values])
ax.set_title('Nível Médio de Estresse por Ocupação', fontsize=14, fontweight='bold')
ax.set_ylabel('Estresse médio (1-10)')
ax.set_xlabel('Ocupação')
ax.axhline(df['Stress_Level'].mean(), color='red', linestyle='--',
           label=f'Média geral: {df["Stress_Level"].mean():.2f}')
for bar, val in zip(bars, stress_occ.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', fontsize=10)
ax.legend()
plt.tight_layout()
plt.savefig('05_estresse_ocupacao.png', bbox_inches='tight')
plt.show()
print(f'\n💡 Insight: {stress_occ.idxmax()} tem o maior estresse médio ({stress_occ.max():.2f}), '
      f'enquanto {stress_occ.idxmin()} tem o menor ({stress_occ.min():.2f}).')

---
## 💼 Ato 4 — E a produtividade?
*Sono e estresse realmente impactam o desempenho?*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fatores que Impactam a Produtividade', fontsize=15, fontweight='bold')

# Sono vs Produtividade
axes[0].scatter(df['Sleep_Hours'], df['Work_Productivity_Score'],
                alpha=0.05, color=PALETTE[0], s=5)
# Linha de tendência
z = np.polyfit(df['Sleep_Hours'], df['Work_Productivity_Score'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['Sleep_Hours'].min(), df['Sleep_Hours'].max(), 100)
axes[0].plot(x_line, p(x_line), color='red', linewidth=2, label='Tendência')
corr_sp = df['Sleep_Hours'].corr(df['Work_Productivity_Score'])
axes[0].set_title('Sono vs Produtividade')
axes[0].set_xlabel('Horas de Sono')
axes[0].set_ylabel('Produtividade (1-10)')
axes[0].text(0.05, 0.92, f'r = {corr_sp:.2f}', transform=axes[0].transAxes,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[0].legend()

# Estresse vs Produtividade
stress_prod = df.groupby('Stress_Level')['Work_Productivity_Score'].mean()
axes[1].bar(stress_prod.index, stress_prod.values,
            color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(stress_prod))))
axes[1].set_title('Produtividade Média por Nível de Estresse')
axes[1].set_xlabel('Nível de Estresse')
axes[1].set_ylabel('Produtividade média')
axes[1].set_xticks(stress_prod.index)

plt.tight_layout()
plt.savefig('06_produtividade.png', bbox_inches='tight')
plt.show()
print(f'\n💡 Correlação sono × produtividade: r = {corr_sp:.3f}')
print('Interprete: valores próximos de 1 indicam relação positiva forte, de 0 indicam ausência de relação.')

In [ ]:
# Cafeína como compensação — usuários com pouco sono
df['Low_Sleep'] = df['Sleep_Hours'] < 6

fig, ax = plt.subplots(figsize=(8, 5))
caff_sleep = df.groupby('Low_Sleep')['Caffeine_Intake_Cups'].mean()
bars = ax.bar(['Sono adequado\n(≥ 6h)', 'Sono insuficiente\n(< 6h)'],
              caff_sleep.values, color=[PALETTE[2], PALETTE[3]], width=0.5)
ax.set_title('Consumo Médio de Cafeína\nSono Adequado vs Insuficiente', fontsize=13, fontweight='bold')
ax.set_ylabel('Xícaras de cafeína/dia')
for bar, val in zip(bars, caff_sleep.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('07_cafeina_sono.png', bbox_inches='tight')
plt.show()
print(f'\n💡 Quem dorme menos usa em média {caff_sleep[True] - caff_sleep[False]:.2f} xícaras a mais por dia — compensação química pelo sono perdido?')

---
## 📋 Resumo dos Principais Insights

In [ ]:
print('=' * 60)
print('  RESUMO — Smartphone, Sono e Estresse')
print('=' * 60)

print(f"""
👥 Perfil:
   • {df.shape[0]:,} usuários, idade média {df['Age'].mean():.0f} anos
   • Dispositivos: {df['Device_Type'].value_counts().to_dict()}

📱 Uso de Tela:
   • Média diária: {df['Daily_Phone_Hours'].mean():.1f}h (máx. {df['Daily_Phone_Hours'].max():.0f}h)
   • Redes sociais: {df['Social_Media_Hours'].mean():.1f}h/dia
   • Fim de semana: {df['Weekend_Screen_Time_Hours'].mean():.1f}h/dia

😴 Saúde:
   • Sono médio: {df['Sleep_Hours'].mean():.1f}h (recomendado: 7-9h)
   • {(df['Sleep_Hours'] < 7).mean()*100:.1f}% dormem menos de 7h
   • Estresse médio: {df['Stress_Level'].mean():.1f}/10

💼 Produtividade:
   • Score médio: {df['Work_Productivity_Score'].mean():.1f}/10
   • Correlação sono × produtividade: {df['Sleep_Hours'].corr(df['Work_Productivity_Score']):.3f}
   • Correlação estresse × produtividade: {df['Stress_Level'].corr(df['Work_Productivity_Score']):.3f}
""")
print('=' * 60)
print('\n🚀 Próximos passos sugeridos:')
print('   • Teste estatístico (ANOVA) entre grupos de ocupação')
print('   • Segmentação com K-Means para criar perfis de usuários')
print('   • Regressão para prever produtividade')